# Week 2: Compare RNN, GRU, and LSTM on IMDb-Tiny

You will compare three recurrent classifiers on **2,000 IMDb reviews: 1,000 negative and 1,000 positive**. You will use 1,600 reviews for training and a separate, balanced set of 400 for validation. Both come from IMDb's official training split; you will not use the official test split in this exercise.

You will run complete, supplied code to:
1. Clean reviews, tokenize them, and build a vocabulary using training text only.
2. Train RNN, GRU, and LSTM classifiers for three epochs each.
3. Compare training and validation curves, training times, parameter counts, and inference timings.
4. Inspect validation reviews misclassified by the RNN but correctly classified by a gated model.
5. Write your own reflection of approximately 150 words in the final markdown cell.

**No coding TODOs remain.** Your assessed work is to inspect the outputs and justify your deployment choice. All three models use the same data, vocabulary, batch order, embedding dimension (100), hidden dimension (128), optimizer settings, and starting embedding/classifier weights. Their recurrent cells differ, so their parameter counts differ.

This is a small, single-seed teaching comparison. No architecture is guaranteed to win. Your deployment choice is provisional: a production decision would require an independent test set and repeated runs.


## 1 — Environment and configuration
Colab commonly provides PyTorch. If an import fails, run this command in a new notebook cell, then restart the kernel if prompted:

```python
%pip install torch pandas numpy matplotlib scikit-learn datasets
```

Downloading the public IMDb data does **not** require a Hugging Face account or token. The first download requires internet access; subsequent runs reuse a local CSV. CPU is sufficient, though runtimes depend on your hardware. You do not need to pin `fsspec` or install `gcsfs`.


In [ ]:
import os
import sys
import time
import copy
import random
import re
import html
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.model_selection import train_test_split

SEED = 42
BATCH_SIZE = 32
MAX_LEN = 200
MAX_VOCAB = 10000
MIN_FREQ = 2
EMB, HID, EPOCHS = 100, 128, 3
LR = 2e-3
PAD_ID, UNK_ID = 0, 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Small CPU workloads may slow down with too many threads.
torch.set_num_threads(min(2, os.cpu_count() or 1))

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
print("Device:", DEVICE, "CPU threads:", torch.get_num_threads())
print(f"Each model: {EPOCHS} epochs; batch={BATCH_SIZE}; embedding={EMB}; hidden={HID}; max tokens={MAX_LEN}")


## 2 — Load a genuinely balanced 2,000-review subset
The original starter added two text Series and two label Series. That concatenated review pairs and added their labels rather than stacking 2,000 examples. Here you explicitly sample by **actual label**, never by assuming that a slice contains positives or negatives.

The new cache filename avoids silently reusing a malformed `imdb_tiny.csv` from the old notebook. If you supply a CSV manually, it must contain `text` and `label` columns with 0 = negative and 1 = positive. The checks require exactly 1,000 unique reviews of each class. Exact duplicate reviews are removed before sampling to prevent overlap across your later split.


In [ ]:
DATA_PATH = Path("imdb_tiny_balanced_2000_v2.csv")
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    from datasets import load_dataset
    print("Downloading/loading the public IMDb training split...")
    # Restrict download to the official training parquet; do not load test/unsupervised data.
    source = load_dataset(
        "stanfordnlp/imdb",
        data_files={"train": "plain_text/train-00000-of-00001.parquet"},
        split="train",
    ).to_pandas()[["text", "label"]]
    source = source.drop_duplicates(subset="text").reset_index(drop=True)
    pieces = [source.loc[source["label"] == label].sample(n=1000, random_state=SEED)
              for label in (0, 1)]
    df = pd.concat(pieces, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    df.to_csv(DATA_PATH, index=False)

if not {"text", "label"}.issubset(df.columns):
    raise ValueError("Your CSV must contain text and label columns.")
if df[["text", "label"]].isna().any().any():
    raise ValueError("The CSV contains missing reviews or labels.")
if not df["text"].map(lambda x: isinstance(x, str) and bool(x.strip())).all():
    raise ValueError("Every review must be a nonempty string.")
df["label"] = pd.to_numeric(df["label"], errors="raise")
if set(df["label"].unique()) != {0, 1}:
    raise ValueError("IMDb labels must be 0 (negative) and 1 (positive).")
df["label"] = df["label"].astype(int)
if len(df) != 2000 or df["label"].value_counts().to_dict() != {0: 1000, 1: 1000}:
    raise ValueError("Expected exactly 2,000 reviews, with 1,000 of each label. Remove the invalid cache and rerun.")
if df["text"].duplicated().any():
    raise ValueError("Duplicate reviews found; regenerate the balanced cache before splitting.")
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
assert set(train_df["text"]).isdisjoint(set(val_df["text"]))
print("Total:", len(df), "Label counts:", df["label"].value_counts().sort_index().to_dict())
print("Training:", len(train_df), train_df["label"].value_counts().sort_index().to_dict())
print("Validation:", len(val_df), val_df["label"].value_counts().sort_index().to_dict())


## 3 — Reuse cleaning and tokenization; freeze the vocabulary
The supplied reference functions retain negation and punctuation. You can replace them with your Week 1 functions if they return token **strings**. Your training results will then change; use the same functions for every model.

You count tokens on the training split only. You reserve 0 for padding and 1 for unknown tokens. Validation encoding never adds vocabulary entries. **Averaging integer token IDs is invalid**: the resulting number would refer to an unrelated vocabulary entry, not an averaged meaning.

For a short practical run, you keep the **first 200 tokens** of each review. This loses information, including possible conclusions at the end of a review. All three models receive the same truncated input, and you should consider this limitation when interpreting errors.


In [ ]:
def clean_text(text):
    text = html.unescape(text)
    text = re.sub(r"<[^>]*>", " ", text)
    text = unicodedata.normalize("NFC", text).lower()
    return re.sub(r"\s+", " ", text).strip()

def my_tokenizer(text):
    return re.findall(r"\w+(?:['’]\w+)*|[^\w\s]", text, flags=re.UNICODE)

train_tokens = [my_tokenizer(clean_text(t)) for t in train_df["text"]]
val_tokens = [my_tokenizer(clean_text(t)) for t in val_df["text"]]
# Count only the tokens that the model can actually receive.
counts = Counter(token for row in train_tokens for token in row[:MAX_LEN])
word_to_id = {"<pad>": PAD_ID, "<unk>": UNK_ID}
for token, freq in sorted(counts.items(), key=lambda item: (-item[1], item[0])):
    if freq >= MIN_FREQ and token not in word_to_id and len(word_to_id) < MAX_VOCAB:
        word_to_id[token] = len(word_to_id)

def encode_tokens(token_list):
    ids = [word_to_id.get(token, UNK_ID) for token in token_list[:MAX_LEN]]
    return ids or [UNK_ID]

vocab_before = len(word_to_id)
train_ids = [encode_tokens(row) for row in train_tokens]
val_ids = [encode_tokens(row) for row in val_tokens]
assert len(word_to_id) == vocab_before
assert encode_tokens([]) == [UNK_ID]
assert encode_tokens(["never_seen_probe_xyz987"]) == [UNK_ID]
assert all(0 < token_id < len(word_to_id) for row in train_ids + val_ids for token_id in row)
print("Vocabulary size:", len(word_to_id))
print("Training reviews truncated:", sum(len(row) > MAX_LEN for row in train_tokens))
print("Validation reviews truncated:", sum(len(row) > MAX_LEN for row in val_tokens))
print("Validation unknown-token rate:", f"{sum(row.count(UNK_ID) for row in val_ids) / sum(map(len, val_ids)):.1%}")


## 4 — Build your PyTorch data pipeline
This follows the previous activity: `Dataset` retrieves one review, `collate_batch` pads a list of examples, and `DataLoader` shuffles and batches them.

You return a boolean mask, then derive real sequence lengths from it in the model. Labels and IDs use `torch.long`. The validation loader never shuffles, so prediction row *i* corresponds to validation review *i*.


In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, sequences, labels):
        assert len(sequences) == len(labels)
        self.sequences = sequences
        self.labels = list(labels)
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        return {"input_ids": self.sequences[idx], "label": self.labels[idx]}

def collate_batch(batch):
    sequences = [torch.tensor(row["input_ids"], dtype=torch.long) for row in batch]
    labels = torch.tensor([row["label"] for row in batch], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True, padding_value=PAD_ID)
    return padded, padded != PAD_ID, labels

train_ds = ReviewDataset(train_ids, train_df["label"].tolist())
val_ds = ReviewDataset(val_ids, val_df["label"].tolist())

def make_train_loader():
    # Reset for each architecture so the shuffled batch order is identical.
    return DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                      collate_fn=collate_batch, generator=torch.Generator().manual_seed(SEED))

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, collate_fn=collate_batch)
ids, mask, labels = next(iter(make_train_loader()))
assert ids.shape == mask.shape and ids.dtype == labels.dtype == torch.long
assert mask.dtype == torch.bool and torch.equal(mask, ids != PAD_ID)
assert (mask.sum(dim=1) > 0).all().item()
print("Example batch:", tuple(ids.shape), "Mask:", tuple(mask.shape), "Labels:", tuple(labels.shape))


## 5 — Define the three classifiers
Each model contains an embedding layer, one unidirectional recurrent layer, and a linear classifier with two output scores. Cross-entropy loss consumes those scores directly; no softmax is needed in the training code.

`pack_padded_sequence` uses the true lengths so the recurrent layer skips padding. The final hidden state represents each review after its last real token. An LSTM returns a hidden state and a cell state; you use the hidden state for classification.

All models start with identical embedding and classifier weights. You hold dimensions constant, **not parameter budgets**: GRU and LSTM gates add parameters. The summary below reports that difference.


In [ ]:
class RecurrentClassifier(nn.Module):
    def __init__(self, cell):
        super().__init__()
        self.embedding = nn.Embedding(len(word_to_id), EMB, padding_idx=PAD_ID)
        recurrent_classes = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}
        if cell not in recurrent_classes:
            raise ValueError(f"Unknown architecture: {cell}")
        self.recurrent = recurrent_classes[cell](EMB, HID, batch_first=True)
        self.classifier = nn.Linear(HID, 2)

    def forward(self, ids, mask):
        lengths = mask.sum(dim=1).to(device="cpu", dtype=torch.long)
        embedded = self.embedding(ids)
        packed = pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)
        _, state = self.recurrent(packed)
        hidden = state[0] if isinstance(state, tuple) else state
        return self.classifier(hidden[-1])

models = {}
for name in ("RNN", "GRU", "LSTM"):
    seed_everything(SEED)
    models[name] = RecurrentClassifier(name).to(DEVICE)
with torch.no_grad():
    for name in ("GRU", "LSTM"):
        models[name].embedding.load_state_dict(models["RNN"].embedding.state_dict())
        models[name].classifier.load_state_dict(models["RNN"].classifier.state_dict())

for name, model in models.items():
    assert torch.equal(model.embedding.weight, models["RNN"].embedding.weight)
    params = sum(p.numel() for p in model.parameters())
    with torch.no_grad():
        example_logits = model(ids.to(DEVICE), mask.to(DEVICE))
        # Adding right padding must not change a packed model's predictions.
        extra_ids = torch.nn.functional.pad(ids, (0, 4), value=PAD_ID)
        extra_mask = torch.nn.functional.pad(mask, (0, 4), value=False)
        padded_logits = model(extra_ids.to(DEVICE), extra_mask.to(DEVICE))
    assert example_logits.shape == (len(labels), 2)
    assert torch.allclose(example_logits, padded_logits, atol=1e-5)
    print(f"{name}: {params:,} trainable parameters; padding-invariance check passed.")


## 6 — Train for three epochs and evaluate on unseen validation reviews
For each epoch, you print training loss/accuracy, validation loss/accuracy, and **training-only epoch time**. The timer excludes validation, checkpoint copying, data downloading, and preprocessing. CUDA synchronization makes GPU timings meaningful.

You clip gradients to reduce exploding-gradient problems. You keep the checkpoint with the highest validation accuracy for later error analysis and inference timing (earliest epoch wins ties). Validation accuracy guides model selection; it is not an unbiased final test score.


In [ ]:
criterion = nn.CrossEntropyLoss()

def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    predictions = []
    for ids, mask, labels in loader:
        ids, mask, labels = ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
        logits = model(ids, mask)
        loss_sum += criterion(logits, labels).item() * len(labels)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += len(labels)
        predictions.extend(preds.cpu().tolist())
    return loss_sum / total, correct / total, np.array(predictions)

history, best_epochs = {}, {}
for name, model in models.items():
    seed_everything(SEED)
    train_loader = make_train_loader()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    history[name] = {key: [] for key in ("train_loss", "train_acc", "val_loss", "val_acc", "train_seconds")}
    best_acc, best_state = -1.0, None
    print(f"\nTraining {name}", flush=True)
    for epoch in range(1, EPOCHS + 1):
        model.train()
        synchronize()
        start = time.perf_counter()
        loss_sum, correct, total = 0.0, 0, 0
        for ids, mask, labels in train_loader:
            ids, mask, labels = ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(ids, mask)
            loss = criterion(logits, labels)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss in {name} epoch {epoch}.")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            loss_sum += loss.item() * len(labels)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += len(labels)
        synchronize()
        elapsed = time.perf_counter() - start
        assert total == len(train_ds)
        val_loss, val_acc, _ = evaluate(model, val_loader)
        values = (loss_sum / total, correct / total, val_loss, val_acc, elapsed)
        for key, value in zip(history[name], values):
            history[name][key].append(value)
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epochs[name] = epoch
        print(f"  Epoch {epoch}/{EPOCHS} | train loss={values[0]:.4f} acc={values[1]:.1%}"
              f" | val loss={val_loss:.4f} acc={val_acc:.1%} | train time={elapsed:.2f}s", flush=True)
    model.load_state_dict(best_state)
    print(f"Restored {name} checkpoint: epoch {best_epochs[name]}, validation accuracy {best_acc:.1%}.")


## 7 — Inspect learning curves and epoch runtimes
The left plot separates training accuracy (dashed) from validation accuracy (solid). The middle plot shows validation loss. The right plot reports training-only epoch times. Epoch numbering starts at 1.

You should consider whether a model's training accuracy improves while its validation performance stalls or worsens. That gap may indicate overfitting; it does not establish the cause by itself.


In [ ]:
epochs = np.arange(1, EPOCHS + 1)
colors = {"RNN": "#2563eb", "GRU": "#d97706", "LSTM": "#15803d"}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for name, values in history.items():
    color = colors[name]
    axes[0].plot(epochs, values["train_acc"], "--", color=color, label=f"{name} train")
    axes[0].plot(epochs, values["val_acc"], "-o", color=color, label=f"{name} validation")
    axes[1].plot(epochs, values["val_loss"], "-o", color=color, label=name)
    axes[2].plot(epochs, values["train_seconds"], "-o", color=color, label=name)
axes[0].axhline(0.5, color="gray", linewidth=1, linestyle=":", label="50% baseline")
axes[0].set(title="Training and validation accuracy", ylabel="Accuracy", ylim=(0, 1))
axes[1].set(title="Validation loss", ylabel="Cross-entropy")
axes[2].set(title=f"Training runtime ({DEVICE})", ylabel="Seconds per epoch")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.set_xticks(epochs)
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
fig.suptitle("IMDb-Tiny: 1,600 training / 400 validation reviews", fontsize=13)
fig.tight_layout()
plt.show()


## 8 — Inspect aligned validation errors
You evaluate the restored best checkpoints on the **same ordered validation set**, so comparisons refer to the same reviews. The code prints up to two examples for each gated model that fixes an RNN error, plus counts of reverse cases where the RNN is correct and the gated model is wrong.

Each excerpt shows the beginning of the actual tokenized model input; the full visible input is at most 200 tokens. Correcting a classification does **not** prove that a model understood a long-distance dependency. Read the text before suggesting an explanation.

If a run produces no corrected RNN errors, report that observation honestly rather than invent an example.


In [ ]:
truth = val_df["label"].to_numpy()
predictions = {}
for name, model in models.items():
    _, accuracy, predictions[name] = evaluate(model, val_loader)
    assert len(predictions[name]) == len(truth)
    assert np.isclose(accuracy, max(history[name]["val_acc"]))
    print(f"{name}: {int((predictions[name] == truth).sum())}/{len(truth)} validation predictions correct.")

label_name = {0: "negative", 1: "positive"}
rnn_wrong = predictions["RNN"] != truth
corrected_indices = {}
for name in ("GRU", "LSTM"):
    fixed = np.flatnonzero(rnn_wrong & (predictions[name] == truth))
    reverse = np.flatnonzero((~rnn_wrong) & (predictions[name] != truth))
    corrected_indices[name] = fixed
    print(f"\n{name}: fixes {len(fixed)} RNN errors; introduces {len(reverse)} errors where RNN is correct.")
    if not len(fixed):
        print("No corrected RNN errors in this run. Use this finding in your reflection.")
    for idx in fixed[:2]:
        visible_tokens = val_tokens[idx][:MAX_LEN]
        excerpt = " ".join(visible_tokens[:90])
        print(f"  Validation row {idx} | true={label_name[int(truth[idx])]}"
              f" | RNN={label_name[int(predictions['RNN'][idx])]}"
              f" | {name}={label_name[int(predictions[name][idx])]}")
        print("  Model-input excerpt:", excerpt)
        if len(visible_tokens) > 90:
            print(f"  [Excerpt shortened; model received {len(visible_tokens)} tokens.]")


## 9 — Compare deployment trade-offs
Training time is not inference latency. You therefore measure a small **model-only batch-inference benchmark** on the same validation batch for every architecture, after warm-up. Report the median over ten repetitions.

- `batch_latency_ms` is the time for one batch on the printed device. It includes packing and device synchronization; it excludes tokenization, data transfer into the initial benchmark batch, networking, and service overhead.
- Dividing batch latency by its size gives an amortized cost per review, **not single-request latency**.
- `parameter_MiB` counts parameter storage at the current dtype. It excludes activations, optimizer states, framework overhead, and working memory; it is not peak RAM.
- Hardware, batching, maximum review length, and software kernels affect the results. Gate count alone cannot predict which implementation runs fastest.

Use this evidence to justify a provisional choice rather than automatically selecting the model with the highest training accuracy.


In [ ]:
bench_ids, bench_mask, _ = next(iter(val_loader))
bench_ids, bench_mask = bench_ids.to(DEVICE), bench_mask.to(DEVICE)

@torch.inference_mode()
def benchmark_batch(model, repetitions=10):
    model.eval()
    for _ in range(3):
        model(bench_ids, bench_mask)
    synchronize()
    times = []
    for _ in range(repetitions):
        synchronize()
        start = time.perf_counter()
        model(bench_ids, bench_mask)
        synchronize()
        times.append((time.perf_counter() - start) * 1000)
    return float(np.median(times))

rows = []
for name, model in models.items():
    params = sum(p.numel() for p in model.parameters())
    parameter_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    batch_ms = benchmark_batch(model)
    rows.append({
        "model": name,
        "best_epoch": best_epochs[name],
        "best_val_accuracy": max(history[name]["val_acc"]),
        "mean_train_epoch_s": np.mean(history[name]["train_seconds"]),
        "total_train_s": sum(history[name]["train_seconds"]),
        "parameters": params,
        "parameter_MiB": parameter_bytes / (1024 ** 2),
        "batch_latency_ms": batch_ms,
        "amortized_ms_per_review": batch_ms / len(bench_ids),
    })
summary = pd.DataFrame(rows).set_index("model")
print(f"Benchmark device={DEVICE}; batch size={len(bench_ids)}; padded width={bench_ids.shape[1]}")
print(summary.round(4).to_string())
print("\nSingle-seed validation comparison: select provisionally; confirm on independent test data before deployment.")


## Submission checklist and references
Restart your kernel and run all cells in order. Submit your completed notebook with:
- Logs for all nine training epochs (three per model), including validation metrics and runtimes.
- The learning-curve figure and deployment comparison table.
- The printed qualitative error comparisons, or the explicit no-correction message if none occurs.
- Your own reflection in the final markdown cell.

Your outputs depend on hardware and library versions. Small accuracy differences on 400 validation reviews may be unstable; one review corresponds to 0.25 percentage points.

**References:** [Stanford IMDb dataset](https://ai.stanford.edu/~amaas/data/sentiment/); [IMDb on Hugging Face](https://huggingface.co/datasets/stanfordnlp/imdb); [PyTorch recurrent layers](https://pytorch.org/docs/stable/nn.html#recurrent-layers); [pack_padded_sequence](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html).


## Reflection — approximately 150 words

Which model—RNN, GRU, or LSTM—would you provisionally choose for this dataset, and why?

In your response:
- Reference your **validation curves**, at least one accuracy value, and the printed epoch runtimes.
- Describe at least one validation error that a gated model corrected, using the printed review excerpt and row number. If none occurred, state that and discuss what your observed errors show instead.
- Explain how latency and memory constraints affect your choice. Distinguish measured batch latency from single-request latency, and parameter storage from total memory.
- Acknowledge a relevant limitation, such as the small validation set, three-epoch budget, single seed, or truncated reviews.

**Your reflection:**

*Replace this line with your response. There is no predetermined winning model.*
